# GenDR → Knowledge Graph (KG) Builder

**Source:** [GenDR](https://genomics.senescence.info/diet/) — Dietary Restriction Gene Manipulations (`cleaned_gendr_manipulations.csv`)  
**Species covered:** *Saccharomyces cerevisiae*, *Caenorhabditis elegans*, *Drosophila melanogaster*, *Mus musculus*


## Output files
```
Yeast/   Yeast_GenDR_Gene_BioProcess.csv
Cele/    Cele_GenDR_Gene_BioProcess.csv
Droso/   Droso_GenDR_Gene_BioProcess.csv
Mouse/   Mouse_GenDR_Gene_BioProcess.csv
```

---
## 0 · Configuration — edit ONLY these two lines

In [4]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : root folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/gendr"

# ── Derived input paths (do not edit below this line) ────────────────────────
# GO ID ↔ name table
GO_PATH          = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
# ENSEMBL ↔ NCBI gene symbol crosswalk (Human)
ENS2NCBI_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
# NCBI gene_info files — one per species
NCBI_HUMAN_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
NCBI_YEAST_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Saccharomyces_cerevisiae.gene_info")
NCBI_CELE_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info")
NCBI_DROSO_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Drosophila_melanogaster.gene_info")
NCBI_MOUSE_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Mus_musculus.gene_info")
# GenDR dietary restriction manipulations dataset
GENDR_PATH       = os.path.join(BASE_PATH, "gendr/cleaned_gendr_manipulations.csv")

# ── Derived output directories ───────────────────────────────────────────────
OUT_YEAST = os.path.join(OUT_PATH, "Yeast/")
OUT_CELE  = os.path.join(OUT_PATH, "Cele/")
OUT_DROSO = os.path.join(OUT_PATH, "Droso/")
OUT_MOUSE = os.path.join(OUT_PATH, "Mouse/")

for d in [OUT_YEAST, OUT_CELE, OUT_DROSO, OUT_MOUSE]:
    os.makedirs(d, exist_ok=True)

# Final column order — applied consistently to all species outputs
DESIRED_ORDER = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Tail_detail_name',
    'Relation_Source', 'Species'
]

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output root : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output root : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/


---
## 1 · Load GO term reference dictionary

In [5]:
# ── GO ID → term name ─────────────────────────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO_dict              = dict(zip(GO['id'], GO['name']))       # {GO:XXXXXXX: term name}
GO_id_namespace_dict = dict(zip(GO['id'], GO['namespace']))  # {GO:XXXXXXX: namespace}

print(f"Loaded {len(GO_dict):,} GO term mappings")
# Verify the DR GO term we will use is present
print(f"  GO:0061771 → {GO_dict.get('GO:0061771', '*** MISSING — check GO file ***')}")

Loaded 47,995 GO term mappings
  GO:0061771 → response to caloric restriction


---
## 2 · Load NCBI gene reference dictionaries (all species)

| Species | GenDR identifier style | Normalisation |
|---|---|---|
| Human | Gene Symbol | Direct symbol lookup |
| Yeast | Gene Symbol or LocusTag | Symbol → description, with LocusTag fallback |
| C. elegans | LocusTag (`CELE_` prefix) | Strip prefix → Symbol → description |
| Drosophila | LocusTag (`Dmel_` prefix stripped in NCBI dict) | LocusTag → Symbol → description |
| Mouse | Gene Symbol | Direct symbol lookup |

In [6]:
# ── 2a. Human — ENSEMBL crosswalk + NCBI gene_info ────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))  # {symbol: ENS_ID}
del ENS_2NCBI  # free memory

NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)

# GeneID → description
NCBI_ALL_GENEname_dict        = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
# GeneID → symbol
NCBI_ALL_GENEIDname_dict      = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
# Symbol → description  (primary lookup for Human gene validation)
NCBI_Synbol_GENEname_dict     = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
# Description → symbol  (reverse lookup)
NCBI_GENEname__Symbol_dict    = dict(zip(NCBI_ALL_GENE['description'], NCBI_ALL_GENE['Symbol']))

# Expand pipe-separated synonyms into a flat alias lookup
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for aliases, canonical in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in str(aliases).split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = canonical

print(f"Human NCBI: {len(NCBI_Synbol_GENEname_dict):,} symbol → description")
print(f"Human synonyms expanded: {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} aliases")

Human NCBI: 193,352 symbol → description
Human synonyms expanded: 69,564 aliases


In [7]:
# ── 2b. Yeast (Saccharomyces cerevisiae) ──────────────────────────────────────
NCBI_Yeast_gene = pd.read_csv(NCBI_YEAST_PATH, sep='\t')

# silently overwriting the Human GeneID→description dict built above.
# Renamed to 'Yeast_Symbol_2_LocusTag_dict' to avoid collision.
Yeast_Symbol_2_LocusTag_dict   = dict(zip(NCBI_Yeast_gene['Symbol'],   NCBI_Yeast_gene['LocusTag']))
NCBI_GENEname_Description_dict = dict(zip(NCBI_Yeast_gene['Symbol'],   NCBI_Yeast_gene['Full_name_from_nomenclature_authority']))
NCBI_LocusTag_Description_dict = dict(zip(NCBI_Yeast_gene['LocusTag'], NCBI_Yeast_gene['Full_name_from_nomenclature_authority']))

print(f"Yeast: {len(NCBI_GENEname_Description_dict):,} symbol → description, {len(NCBI_LocusTag_Description_dict):,} locustag → description")

Yeast: 6,513 symbol → description, 6,513 locustag → description


In [8]:
# ── 2c. C. elegans ───────────────────────────────────────────────────────────
NCBI_Cele_gene = pd.read_csv(NCBI_CELE_PATH, sep='\t')

# Strip 'CELE_' prefix from LocusTags (e.g. 'CELE_R07B1.9' → 'R07B1.9')
NCBI_Cele_gene['LocusTag'] = NCBI_Cele_gene['LocusTag'].str.replace('CELE_', '', regex=False)

CELE_LocusTag_2_Symbol_dict       = dict(zip(NCBI_Cele_gene['LocusTag'], NCBI_Cele_gene['Symbol']))
NCBI_Cele_Symbol_Description_dict = dict(zip(NCBI_Cele_gene['Symbol'],   NCBI_Cele_gene['description']))

print(f"C. elegans: {len(CELE_LocusTag_2_Symbol_dict):,} locustag → symbol")

C. elegans: 46,927 locustag → symbol


In [9]:
# ── 2d. Drosophila melanogaster ───────────────────────────────────────────────
NCBI_Droso_gene = pd.read_csv(NCBI_DROSO_PATH, sep='\t')

# Strip 'Dmel_' prefix from LocusTags (e.g. 'Dmel_CG5362' → 'CG5362')
NCBI_Droso_gene['LocusTag'] = NCBI_Droso_gene['LocusTag'].str.replace('Dmel_', '', regex=False)

Droso_LocusTag_2_Symbol_dict       = dict(zip(NCBI_Droso_gene['LocusTag'], NCBI_Droso_gene['Symbol']))
NCBI_Droso_Symbol_Description_dict = dict(zip(NCBI_Droso_gene['Symbol'],   NCBI_Droso_gene['description']))

print(f"Drosophila: {len(Droso_LocusTag_2_Symbol_dict):,} locustag → symbol")

Drosophila: 17,873 locustag → symbol


In [10]:
# ── 2e. Mouse (Mus musculus) ──────────────────────────────────────────────────
# Mouse LocusTags have NO 'Dmel_' prefix — this was a copy-paste error from
# the Drosophila cell. Mouse LocusTags are loaded as-is.
NCBI_Mouse_gene = pd.read_csv(NCBI_MOUSE_PATH, sep='\t')

NCBI_Mouse_Symbol_Description_dict = dict(zip(NCBI_Mouse_gene['Symbol'], NCBI_Mouse_gene['description']))

print(f"Mouse: {len(NCBI_Mouse_Symbol_Description_dict):,} symbol → description")

Mouse: 112,035 symbol → description


---
## 3 · Load and prepare GenDR data

**Source:** `cleaned_gendr_manipulations.csv`  
**GO term assigned:** `GO:0061771` — *response to dietary restriction*  

All GenDR genes mediate the lifespan effect of dietary restriction.  
A single relation type `Gene_Biological_process` is used (no Inhibits/Promotes split)  
because GenDR records **which genes are required for DR** to work, not the direction of aging effect.

In [11]:
# ── 3a. Load GenDR — skip malformed lines gracefully ──────────────────────────

#   enDR = pd.read_csv(file_path, error_bad_lines=False, warn_bad_lines=True, ...)
# 'enDR' is not 'GenDR' (missing 'G'), and 'error_bad_lines' was removed in pandas 2.0.
# Removed entirely below.
GenDR = pd.read_csv(GENDR_PATH, on_bad_lines='skip', engine='python')

# Strip leading/trailing whitespace from gene symbols (present in source data)
GenDR['gene symbol'] = GenDR['gene symbol'].str.strip()
GenDR = GenDR.rename(columns={'gene symbol': 'Head', 'species': 'Species'})

print(f"Loaded {len(GenDR):,} rows")
print("Species distribution:")
print(GenDR['Species'].value_counts())
GenDR.head(3)

Loaded 214 rows
Species distribution:
Species
Saccharomyces cerevisiae     112
Caenorhabditis elegans        62
Drosophila melanogaster       27
Mus musculus                   7
Schizosaccharomyces pombe      6
Name: count, dtype: int64


,GenDR ID,Head,Species,entrez gene id,gene name
0,1,SIR2,Saccharomyces cerevisiae,851520,Silent Information Regulator 2
1,2,CDC25,Saccharomyces cerevisiae,851019,Cell Division Cycle 25
2,3,HAP4,Saccharomyces cerevisiae,853751,Heme Activator Protein 4


In [12]:
# ── 3b. Assign KG schema columns ─────────────────────────────────────────────
GenDR['Head_type']       = 'Gene'
GenDR['Tail_type']       = 'Biological_process'
# All GenDR genes relate to the dietary restriction biological process GO term
GenDR['Tail']            = 'GO:0061771'
GenDR['Tail_detail_name']= GenDR['Tail'].map(GO_dict)       # 'response to dietary restriction'
GenDR['Relation']        = GenDR['Head_type'] + '_' + GenDR['Tail_type']   # 'Gene_Biological_process'
GenDR['Relation_Source'] = 'GenDR'
GenDR['Tail_ID_IS']      = 'Quick_GO'
# Head_ID_IS is assigned per-species below after gene symbol normalisation

print(f"Tail detail name assigned: {GenDR['Tail_detail_name'].iloc[0]}")
GenDR.head(3)

Tail detail name assigned: response to caloric restriction


,GenDR ID,Head,Species,entrez gene id,gene name,Head_type,Tail_type,Tail,Tail_detail_name,Relation,Relation_Source,Tail_ID_IS
0,1,SIR2,Saccharomyces cerevisiae,851520,Silent Information Regulator 2,Gene,Biological_process,GO:0061771,response to caloric restriction,Gene_Biological_process,GenDR,Quick_GO
1,2,CDC25,Saccharomyces cerevisiae,851019,Cell Division Cycle 25,Gene,Biological_process,GO:0061771,response to caloric restriction,Gene_Biological_process,GenDR,Quick_GO
2,3,HAP4,Saccharomyces cerevisiae,853751,Heme Activator Protein 4,Gene,Biological_process,GO:0061771,response to caloric restriction,Gene_Biological_process,GenDR,Quick_GO


---
## 4 · Per-species processing and export

Each section:
1. Filters `GenDR` to that species
2. Normalises gene identifiers (LocusTag → Symbol where needed)
3. Validates against NCBI description dict (drops unmatched rows)
4. Applies consistent `DESIRED_ORDER` column selection
5. Exports one CSV file

### 4a · Yeast (*Saccharomyces cerevisiae*)

In [13]:
Yeast_GenDR = GenDR[GenDR['Species'] == 'Saccharomyces cerevisiae'].copy()
print(f"Yeast raw: {len(Yeast_GenDR):,} rows")

# Yeast GenDR entries may be Gene Symbols OR LocusTags.
# Try Symbol→description first; fall back to LocusTag→description.
Yeast_GenDR['Head_detail_name'] = (
    Yeast_GenDR['Head'].map(NCBI_GENEname_Description_dict)
    .fillna(Yeast_GenDR['Head'].map(NCBI_LocusTag_Description_dict))
)

# Map Symbol → LocusTag as alternate identifier (kept for traceability)

Yeast_GenDR['Head_alt_name'] = Yeast_GenDR['Head'].map(Yeast_Symbol_2_LocusTag_dict).fillna(Yeast_GenDR['Head'])
Yeast_GenDR['Head_ID_IS']    = 'NCBI_ID'

# Drop rows where neither Symbol nor LocusTag lookup resolved a description
before = len(Yeast_GenDR)
Yeast_GenDR = Yeast_GenDR[~Yeast_GenDR['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Yeast_GenDR):,} kept / {before - len(Yeast_GenDR):,} dropped")
print(Yeast_GenDR['Relation'].value_counts())

Yeast raw: 112 rows
NCBI validated: 110 kept / 2 dropped
Relation
Gene_Biological_process    110
Name: count, dtype: int64


In [14]:
Yeast_GenDR = Yeast_GenDR[[c for c in DESIRED_ORDER if c in Yeast_GenDR.columns]]

out_path = os.path.join(OUT_YEAST, 'Yeast_GenDR_Gene_BioProcess.csv')
Yeast_GenDR.to_csv(out_path, index=False)
print(f"Saved {len(Yeast_GenDR):,} rows → {out_path}")

Saved 110 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Yeast/Yeast_GenDR_Gene_BioProcess.csv


### 4b · C. elegans (*Caenorhabditis elegans*)

In [15]:
Celeg_GenDR = GenDR[GenDR['Species'] == 'Caenorhabditis elegans'].copy()
print(f"C. elegans raw: {len(Celeg_GenDR):,} rows")

# GenDR C. elegans entries may have 'CELE_' prefix on LocusTags — strip it
Celeg_GenDR['Head'] = Celeg_GenDR['Head'].str.replace('CELE_', '', regex=False)

# Replaced with a clean separate column.
Celeg_GenDR['Head_alt_name'] = Celeg_GenDR['Head']   # save original LocusTag
Celeg_GenDR['Head'] = (
    Celeg_GenDR['Head'].map(CELE_LocusTag_2_Symbol_dict)
    .fillna(Celeg_GenDR['Head'])   # keep LocusTag if no symbol match found
)
Celeg_GenDR['Head_ID_IS'] = 'NCBI_ID'

# Validate Symbol against NCBI description dict
Celeg_GenDR['Head_detail_name'] = Celeg_GenDR['Head'].map(NCBI_Cele_Symbol_Description_dict)
before = len(Celeg_GenDR)
Celeg_GenDR = Celeg_GenDR[~Celeg_GenDR['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Celeg_GenDR):,} kept / {before - len(Celeg_GenDR):,} dropped")
print(Celeg_GenDR['Relation'].value_counts())

C. elegans raw: 62 rows
NCBI validated: 60 kept / 2 dropped
Relation
Gene_Biological_process    60
Name: count, dtype: int64


In [16]:

Celeg_GenDR = Celeg_GenDR[[c for c in DESIRED_ORDER if c in Celeg_GenDR.columns]]

out_path = os.path.join(OUT_CELE, 'Cele_GenDR_Gene_BioProcess.csv')
Celeg_GenDR.to_csv(out_path, index=False)
print(f"Saved {len(Celeg_GenDR):,} rows → {out_path}")

Saved 60 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Cele/Cele_GenDR_Gene_BioProcess.csv


### 4c · Drosophila (*Drosophila melanogaster*)

In [17]:
Droso_GenDR = GenDR[GenDR['Species'] == 'Drosophila melanogaster'].copy()
print(f"Drosophila raw: {len(Droso_GenDR):,} rows")

# Map LocusTag → canonical NCBI Symbol (Dmel_ prefix already stripped in dict construction).

Droso_GenDR['Head_alt_name'] = Droso_GenDR['Head']   # save original LocusTag
Droso_GenDR['Head'] = (
    Droso_GenDR['Head'].map(Droso_LocusTag_2_Symbol_dict)
    .fillna(Droso_GenDR['Head'])
)
Droso_GenDR['Head_ID_IS'] = 'NCBI_ID'

# Validate Symbol against NCBI description dict
Droso_GenDR['Head_detail_name'] = Droso_GenDR['Head'].map(NCBI_Droso_Symbol_Description_dict)
before = len(Droso_GenDR)
Droso_GenDR = Droso_GenDR[~Droso_GenDR['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Droso_GenDR):,} kept / {before - len(Droso_GenDR):,} dropped")
print(Droso_GenDR['Relation'].value_counts())

Drosophila raw: 27 rows
NCBI validated: 21 kept / 6 dropped
Relation
Gene_Biological_process    21
Name: count, dtype: int64


In [18]:
Droso_GenDR = Droso_GenDR[[c for c in DESIRED_ORDER if c in Droso_GenDR.columns]]

out_path = os.path.join(OUT_DROSO, 'Droso_GenDR_Gene_BioProcess.csv')
Droso_GenDR.to_csv(out_path, index=False)
print(f"Saved {len(Droso_GenDR):,} rows → {out_path}")

Saved 21 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Droso/Droso_GenDR_Gene_BioProcess.csv


### 4d · Mouse (*Mus musculus*)

In [19]:
Mouse_GenDR = GenDR[GenDR['Species'] == 'Mus musculus'].copy()
print(f"Mouse raw: {len(Mouse_GenDR):,} rows")

# Mouse GenDR uses gene symbols directly — no LocusTag normalisation needed.
# Validate against Mouse NCBI description dict.
Mouse_GenDR['Head_ID_IS']      = 'NCBI_ID'
Mouse_GenDR['Head_detail_name'] = Mouse_GenDR['Head'].map(NCBI_Mouse_Symbol_Description_dict)
before = len(Mouse_GenDR)
Mouse_GenDR = Mouse_GenDR[~Mouse_GenDR['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Mouse_GenDR):,} kept / {before - len(Mouse_GenDR):,} dropped")
print(Mouse_GenDR['Relation'].value_counts())

Mouse raw: 7 rows
NCBI validated: 7 kept / 0 dropped
Relation
Gene_Biological_process    7
Name: count, dtype: int64


In [20]:

Mouse_GenDR = Mouse_GenDR[[c for c in DESIRED_ORDER if c in Mouse_GenDR.columns]]

out_path = os.path.join(OUT_MOUSE, 'Mouse_GenDR_Gene_BioProcess.csv')
Mouse_GenDR.to_csv(out_path, index=False)
print(f"Saved {len(Mouse_GenDR):,} rows → {out_path}")

Saved 7 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Mouse/Mouse_GenDR_Gene_BioProcess.csv


---
## 5 · Summary — all output files written

In [21]:
print(f"Output files under: {OUT_PATH}\n")
total = 0
for root, dirs, files in os.walk(OUT_PATH):
    dirs.sort()
    for fname in sorted(files):
        if fname.endswith('.csv') and 'GenDR' in fname:
            full = os.path.join(root, fname)
            rows = sum(1 for _ in open(full)) - 1
            size = os.path.getsize(full)
            rel  = os.path.relpath(full, OUT_PATH)
            total += 1
            print(f"  {rel:<60s}  {rows:>5,} rows  {size:>8,} bytes")
print(f"\nTotal: {total} files")

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/

  Cele/Cele_GenDR_Gene_BioProcess.csv                              60 rows    10,395 bytes
  Droso/Droso_GenDR_Gene_BioProcess.csv                            21 rows     3,158 bytes
  Mouse/Mouse_GenDR_Gene_BioProcess.csv                             7 rows     1,071 bytes
  Yeast/Yeast_GenDR_Gene_BioProcess.csv                           110 rows    17,208 bytes

Total: 4 files
